# 10 · Agente con herramientas: diseño, validación y evaluación (T-15 y T-16, bloque N)

**Dónde:** `api/_lib/agente/herramientas.py` (herramientas y esquemas), `api/_lib/agente/agente.py` (router y
dispatcher), `pipeline/src/agents/evaluar.py` (métricas). **Por qué:** una consulta real del área de atención suele
necesitar varios datos ("¿cuánto debe esta toma y pagará a tiempo?"). **Cómo:** en cada turno el LLM elige UNA
herramienta con argumentos en JSON; el dispatcher valida la decisión y los argumentos (Pydantic), ejecuta, mide el
tiempo y devuelve la observación; máximo 5 pasos; cada paso se registra en `agente.log`. **Datos:** Supabase
(`raw.recibos`, `analitica.predicciones_pago`, resultados de series) y el tarifario CEA.

In [ ]:
import sys, pathlib
raiz = pathlib.Path.cwd()
while not (raiz / "pipeline").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from pipeline.notebooks.graficas import dibujar, tabla

In [ ]:
import json
def resultado(modulo, clave):
    """Payload publicado por el evaluador (pipeline/artefactos/resultados/<modulo>.json) o None."""
    ruta = raiz / "pipeline" / "artefactos" / "resultados" / f"{modulo}.json"
    if not ruta.exists():
        return None
    return next((f["payload"] for f in json.loads(ruta.read_text(encoding="utf-8")) if f["clave"] == clave), None)

In [ ]:
from api._lib.agente.herramientas import HERRAMIENTAS
pd.DataFrame([{"herramienta": h.nombre, "descripcion": h.descripcion,
               "argumentos": ", ".join(h.esquema()["argumentos"]), "obligatorios": ", ".join(h.esquema()["obligatorios"])}
              for h in HERRAMIENTAS.values()])

## Validación y manejo de errores del dispatcher

Para mostrar el mecanismo **sin gastar cuota del LLM** se usa un LLM guionizado (decisiones fijas) y datos de
ejemplo. Esto demuestra la validación, no mide al agente: la métrica real está al final.

In [ ]:
import json
from api._lib.agente.agente import ejecutar_agente, MAX_PASOS
from api._lib.agente.herramientas import Contexto

class DatosEjemplo:
    def recibos(self, id_toma, n):
        return [{"periodo": "2026-09", "fecha_vencimiento": "2026-09-20", "total_pagar": 700.5, "pagado": False, "pago_tardio": None}] if id_toma == "TEJEMPLO01" else []
    def predicciones(self, id_toma, periodo):
        return [{"periodo": "2026-09", "prob_pago_tardio": 0.8, "clase_predicha": True, "modelo": "regresion_logistica", "version": "v1"}] if id_toma == "TEJEMPLO01" else []
    def resultados(self, modulo):
        return []

class Guion:
    nombre = "guion"
    def __init__(self, decisiones): self.d = list(decisiones)
    def generar_json(self, sistema, usuario): return json.dumps(self.d.pop(0))

ctx = Contexto(datos=DatosEjemplo())
bitacora = []
salida, detalle = ejecutar_agente(
    "¿Cuánto debe la toma TEJEMPLO01 y pagará tarde?", None,
    Guion([{"accion": "herramienta", "herramienta": "calcular_importe", "argumentos": {"tipo_tarifa": "lunar", "consumo_m3": 5}},
           {"accion": "herramienta", "herramienta": "estado_cuenta", "argumentos": {"id_toma": "TEJEMPLO01"}},
           {"accion": "herramienta", "herramienta": "predecir_pago", "argumentos": {"id_toma": "TEJEMPLO01"}},
           {"accion": "responder", "respuesta": "Debe 700.50 y está en riesgo de pagar tarde."}]),
    ctx, bitacora.append)
pd.DataFrame([{"paso": b["paso"], "herramienta": b["herramienta"], "error": b["error"], "ms": b["ms"]} for b in bitacora])

In [ ]:
print("Respuesta:", salida.respuesta)
print("Errores de herramienta:", detalle["errores_herramienta"], "· decisiones inválidas:", detalle["decisiones_invalidas"])
limite, _ = ejecutar_agente("repite", None, Guion([{"accion": "herramienta", "herramienta": "estado_cuenta", "argumentos": {"id_toma": "TEJEMPLO01"}}] * 7), ctx)
print(f"Con un LLM que nunca responde, el agente se detiene en {len(limite.pasos)} pasos (límite {MAX_PASOS}): {limite.respuesta[:80]}…")

## Casos de evaluación (T-16b)

Borrador en `docs/eval/agente_casos_borrador.csv` con la(s) herramienta(s) esperada(s); **solo los validados por
Andrés** cuentan. La evaluación llama a la URL pública (donde viven las llaves) con `python -m pipeline.src.agents.evaluar`.

In [ ]:
from pipeline.src.agents.evaluar import cargar_casos
casos = cargar_casos(incluir_sin_validar=True)
print(f"{len(casos)} casos · {sum(c['multipaso'] for c in casos)} multipaso · {len(cargar_casos())} validados")
pd.Series([h for c in casos for h in c["esperadas"]]).value_counts().rename("veces esperada")

## Resultado: Tool Selection Accuracy y éxito de tareas (agente_eval)

In [ ]:
p = resultado("agente_eval", "tool_selection")
if p:
    dibujar(p)
else:
    print("PENDIENTE: aún no se corre `python -m pipeline.src.agents.evaluar` (requiere casos validados y /api/agente desplegado).")

## Conclusiones

In [ ]:
print(f"1. El agente tiene {len(HERRAMIENTAS)} herramientas con argumentos validados; un argumento inválido vuelve al LLM como observación y no rompe la respuesta.")
print(f"2. El límite de {MAX_PASOS} pasos se cumple aunque el LLM nunca decida responder, y cada paso queda en agente.log con su tiempo.")
print(f"3. Hay {len(casos)} casos de evaluación ({sum(c['multipaso'] for c in casos)} de varios pasos) escritos antes de correr al agente.")
print("4. " + (p["conclusion"] if p else "La Tool Selection Accuracy está pendiente de la validación humana de los casos; no se reporta un número sin correrla."))